In [1]:
using Pkg
Pkg.activate("../.")
Pkg.instantiate()

  Activating project at `~/Documents/lung_ECM_TDA_new`


In [ ]:
include("../src/ECM_TDA.jl")
using .ECM_TDA

include("../src/Eirene_var.jl")
using .Eirene_var
using Ripserer
using CSV
using DataFrames
using DelimitedFiles
using Plots
using Plots.PlotMeasures
using JLD2
using PersistenceDiagrams


WebIO._IJuliaInit()

Error processing line 1 of /Users/hyoon-24/anaconda3/lib/python3.11/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "<frozen site>", line 186, in addpackage
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored


# 1. on ECM

1(a) Compute persistent homology on ECM point cloud

In [3]:
dir = "../data/pointcloud/ECM/"
csv_files = [item for item in walkdir(dir)][1][3:end][1];

PD0_ECM = Dict()
PD1_ECM = Dict()

for file in csv_files
    filename = split(file, ".")[1]

    df = CSV.read(dir * file, DataFrame)
    if size(df,1) != 0
        PD0, PD1 = run_PH(df)
    
        # save
        writedlm("persistent_homology_outputs/ECM/persistence_diagrams/PD0/" * filename * ".csv", PD0, ",")
        writedlm("persistent_homology_outputs/ECM/persistence_diagrams/PD1/" * filename * ".csv", PD1, ",")
        
        # combine to dictionary
        PD0_ECM[filename] = PD0
        PD1_ECM[filename] = PD1
    else
        PD0 = nothing
        PD1 = nothing
        
        writedlm("persistent_homology_outputs/ECM/persistence_diagrams/PD0/" * filename * ".csv", zeros(), ",")
        writedlm("persistent_homology_outputs/ECM/persistence_diagrams/PD1/" * filename * ".csv", zeros(), ",")
        # combine to dictionary
        PD0_ECM[filename] = PD0
        PD1_ECM[filename] = PD1
    end

end

1(b) compute persistence images 

In [15]:
# convert array to Ripserer PD
PH0 = Dict(k => ECM_TDA.array_to_ripsererPD(v) for (k,v) in PD0_ECM if v != nothing)
PH1 = Dict(k => ECM_TDA.array_to_ripsererPD(v) for (k,v) in PD1_ECM if v != nothing)

PI0 = PersistenceImage([PH0[k] for k in keys(PH0)], sigma=50, size = 20)
PI1 = PersistenceImage([PH1[k] for k in keys(PH1)], sigma=50, size = 20)

ECM_PI0 = Dict()
for i in keys(PH0)
    ECM_PI0[i] = PI0(PH0[i])
end

ECM_PI1 = Dict()
for i in keys(PH1)
    ECM_PI1[i] = PI1(PH1[i])
end


In [18]:
# save the persistence images as CSV files 

for file in csv_files 
    ROI = split(file, ".")[1]

    ROI_PI0 = ECM_PI0[ROI]
    ROI_PI1 = ECM_PI1[ROI]
    
    writedlm("persistent_homology_outputs/ECM/persistence_images/PI0/" * file, ROI_PI0, "," )
    writedlm("persistent_homology_outputs/ECM/persistence_images/PI1/" * file, ROI_PI1, "," )
end

save figures of persistence diagrams and persistence images

In [ ]:
# get maximum values (for plotting purposes)
PD0_ECM = Dict(k =>v for (k,v) in PD0_ECM if v != nothing)
PD1_ECM = Dict(k => v for (k,v) in PD1_ECM if v != nothing)
max0 = get_PD0_max(PD0_ECM)
max1 = get_PD1_max(PD1_ECM)

In [33]:
### save plots of persistence diagrams and persistence images

output_dir = "persistent_homology_outputs/ECM/"
# dim 0
for k in keys(PD0_ECM)
    p = histogram(PD0_ECM[k][:,2], xlims = (0, max0), label = "")
    savefig(output_dir * "persistence_diagrams/PD0_figures/" * k * ".png")
end

# dim 1
for k in keys(PD1_ECM)
    p = ECM_TDA.plot_PD(PD1_ECM[k], pd_min = 0, pd_max = max1, frame = :box)
    savefig(output_dir * "persistence_diagrams/PD1_figures/" * k * ".png")
end

### save persistence images
# dim 0
for k in keys(PI0)
    p = heatmap(PI0[k], rightmargin = 10mm, size = (380, 300))
    savefig(output_dir * "persistence_images/PI0_figures/" * k * ".png")
end

# dim 1
for k in keys(PI1)
    p = heatmap(PI1[k], rightmargin = 10mm, size = (380, 300))
    savefig(output_dir * "persistence_images/PI1_figures/" * k * ".png")
end

In [6]:
# save the persistence diagrams and images as JLD file 
save("persistent_homology_outputs/ECM/PD.jld2", 
"PD0_ECM", PD0_ECM,
"PD1_ECM", PD1_ECM)

save("persistent_homology_outputs/ECM/PI.jld2",
"PI0_ECM", PI0,
"PI1_ECM", PI1
)

In [21]:
# save the min, max coordinates of PIs (useful for plotting)
PI0_xmin = PI0.xs[1]
PI0_xmax = PI0.xs[end]
PI0_ymin = PI0.ys[1]
PI0_ymax = PI0.ys[end]

PI1_xmin = PI1.xs[1]
PI1_xmax = PI1.xs[end]
PI1_ymin = PI1.ys[1]
PI1_ymax = PI1.ys[end]

1481.6103572712373

In [22]:
save("persistent_homology_outputs/ECM/PI_ranges.jld2",
    "PI0_xmin", PI0_xmin,
    "PI0_xmax", PI0_xmax,
    "PI0_ymin", PI0_ymin,
    "PI0_ymax", PI0_ymax,
    "PI1_xmin", PI1_xmin,
    "PI1_xmax", PI1_xmax,
    "PI1_ymin", PI1_ymin,
    "PI1_ymax", PI1_ymax)

# 2. on cancer cells and leukocytes
One can proceed similarly as above